In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import pickle
import warnings
from tqdm import tqdm

# Import library untuk Ekstraksi Fitur
from PyEMD import CEEMDAN
from scipy.signal import hilbert
from sklearn.decomposition import FastICA
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')

# ============================================================================
# 1. PENGATURAN PATH
# ============================================================================
# Mengambil direktori saat ini (asumsi notebook berada di src/pipelines/training)
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, '../../../'))

TOKENIZER_PATH = os.path.join(CURRENT_DIR, 'all_eq_3_0_fixed_hilbert_char_tokenizer_1_1.pkl')
MODEL_PATH = os.path.join(CURRENT_DIR, 'all_eq_3_0_fixed_hilbert_best_model_1_1.pt')
TEST_CSV_PATH = os.path.join(PROJECT_ROOT, 'dataset', 'all_eq_3_0_test.csv')
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'dataset', 'raw')
OUTPUT_CSV_PATH = os.path.join(CURRENT_DIR, 'all_eq_3_0_fixed_hilbert_test_predictions_1_1.csv')

# Menambahkan path model ke sys.path agar import berhasil
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src/model'))

from misc.tokenizer import CharTokenizer
import misc.beam_decoder_char as beam_decoder_char
from model import ConformerTransducer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EEG_CHANNELS = ['EEG.AF3', 'EEG.F7', 'EEG.F3', 'EEG.FC5', 'EEG.T7', 
                'EEG.P7', 'EEG.O1', 'EEG.O2', 'EEG.P8', 'EEG.T8', 
                'EEG.FC6', 'EEG.F4', 'EEG.F8', 'EEG.AF4']

# ============================================================================
# 2. FUNGSI UTILITAS & EKSTRAKSI FITUR (HILBERT SPECTRUM)
# ============================================================================

def remove_ocular_artifacts_ica(eeg_signal, ch_names, threshold=0.6):
    frontal_indices = [i for i, ch in enumerate(ch_names) if 'AF3' in ch or 'AF4' in ch]
    if not frontal_indices: return eeg_signal 

    ica = FastICA(n_components=eeg_signal.shape[1], random_state=42, max_iter=1000, tol=0.01)
    try:
        components = ica.fit_transform(eeg_signal) 
    except:
        return eeg_signal 

    bad_components = []
    for i in range(components.shape[1]):
        is_artifact = False
        for f_idx in frontal_indices:
            corr, _ = pearsonr(components[:, i], eeg_signal[:, f_idx])
            if abs(corr) > threshold:
                is_artifact = True; break
        if is_artifact: bad_components.append(i)

    if bad_components: components[:, bad_components] = 0.0
    return ica.inverse_transform(components)

def extract_eeg_channels(eeg_df):
    if all(ch in eeg_df.columns for ch in EEG_CHANNELS):
        return eeg_df[EEG_CHANNELS].values
    raise ValueError("Not all channels found in CSV")

def load_eeg_signal(id_val, subject, gender, config):
    csv_folder = os.path.join(RAW_DATA_PATH, gender, subject, 'csv')
    if not os.path.isdir(csv_folder): return None
    
    # Gunakan str(id_val) untuk mencegah bug Pandas TypeError
    matching_files = [f for f in os.listdir(csv_folder) if f.startswith(str(id_val) + '_') and f.endswith('.bp.csv')]
    if not matching_files: return None
    
    file_path = os.path.join(csv_folder, matching_files[0])
    try:
        df = pd.read_csv(file_path, skiprows=1)
        signal = extract_eeg_channels(df)
        if config.get('remove_eye_artifacts', True) and signal is not None:
            signal = remove_ocular_artifacts_ica(signal, EEG_CHANNELS, config['ica_threshold'])
        return signal
    except Exception as e:
        print(f"[ERROR] Failed to load {file_path}: {e}")
        return None

def compute_hilbert_spectrum(eeg_signal, config):
    n_samples, n_channels = eeg_signal.shape
    fs = config['sample_rate']
    f_min, f_max, n_bins = config['f_min'], config['f_max'], config['n_freq_bins']
    hop_length, win_length = config['hop_length'], config['win_length']
    start_imf = config.get('start_imf', 2)
    
    freq_edges = np.linspace(f_min, f_max, n_bins + 1)
    ceemdan = CEEMDAN(trials=config['ceemdan_trials'], noise_scale=0.2, parallel=False)
    all_channel_spectra = []
    
    for ch_idx in range(n_channels):
        signal = eeg_signal[:, ch_idx].astype(np.float64)
        imfs = ceemdan(signal)
        
        if start_imf < imfs.shape[0]: imfs = imfs[start_imf:]
        else: imfs = imfs[-1:]
        
        current_n_samples = n_samples
        hilbert_spec = np.zeros((n_bins, n_samples))
        
        for i in range(imfs.shape[0]):
            analytic_signal = hilbert(imfs[i])
            amp, phase = np.abs(analytic_signal), np.unwrap(np.angle(analytic_signal))
            freq = np.insert((np.diff(phase) / (2.0*np.pi) * fs), 0, 0)
            
            bin_indices = np.digitize(freq, freq_edges) - 1
            for t in range(n_samples):
                b = bin_indices[t]
                if 0 <= b < n_bins: hilbert_spec[b, t] += (amp[t] ** 2) 
        
        if current_n_samples > win_length:
            rem = (current_n_samples - win_length) % hop_length
            if rem > 0:
                pad = hop_length - rem
                hilbert_spec = np.pad(hilbert_spec, ((0, 0), (0, pad)), mode='constant')
                current_n_samples += pad

        if current_n_samples < win_length:
            n_frames, framed_spec = 0, np.zeros((n_bins, 0)) 
        else:
            n_frames = 1 + (current_n_samples - win_length) // hop_length
            framed_spec = np.zeros((n_bins, n_frames))
            for t_idx in range(n_frames):
                start = t_idx * hop_length
                framed_spec[:, t_idx] = np.mean(hilbert_spec[:, start:start+win_length], axis=1)  
        
        all_channel_spectra.append(framed_spec)
        
    all_channel_spectra = np.array(all_channel_spectra).transpose(2, 0, 1)
    features_flat = np.log(all_channel_spectra.reshape(all_channel_spectra.shape[0], -1) + 1e-9)
    return ((features_flat - np.mean(features_flat, axis=0)) / (np.std(features_flat, axis=0) + 1e-6)).astype(np.float32)

def compute_cer(reference, hypothesis):
    if len(reference) == 0: return 1.0 if len(hypothesis) > 0 else 0.0
    d = np.zeros((len(reference) + 1, len(hypothesis) + 1))
    for i in range(len(reference) + 1): d[i][0] = i
    for j in range(len(hypothesis) + 1): d[0][j] = j
    for i in range(1, len(reference) + 1):
        for j in range(1, len(hypothesis) + 1):
            cost = 0 if reference[i-1] == hypothesis[j-1] else 1
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + cost)
    return d[len(reference)][len(hypothesis)] / len(reference)

# ============================================================================
# 3. PIPELINE INFERENSI
# ============================================================================
print("=" * 80)
print(f"EEG-to-Text Inference Pipeline (All Subjects | Fixed Hilbert)")
print(f"[INFO] Menggunakan device: {DEVICE}") 
print("=" * 80)

# 1. LOAD TOKENIZER
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(f"File Tokenizer tidak ditemukan: {TOKENIZER_PATH}")
print(f"Loading Tokenizer dari: all_eq_3_0_fixed_hilbert_char_tokenizer_1_1.pkl")
with open(TOKENIZER_PATH, 'rb') as f:
    tokenizer = pickle.load(f)

# 2. LOAD MODEL & CONFIG
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"File Model tidak ditemukan: {MODEL_PATH}")
print(f"Loading Model dari: all_eq_3_0_fixed_hilbert_best_model_1_1.pt")
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

# Ambil CONFIG yang disimpan saat training
config = checkpoint['config']
print(f"Config berhasil dimuat. Vocab size: {config.get('vocab_size')}")

# Bangun arsitektur model dan muat state_dict
model = ConformerTransducer(config).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval() # PENTING: Set model ke mode evaluasi

# Inisialisasi Beam Decoder
beam_decoder = beam_decoder_char.BeamDecoderChar(model, tokenizer, beam_size=3, max_sym_per_frame=15)

# 3. LOAD DATASET TEST
if not os.path.exists(TEST_CSV_PATH):
    raise FileNotFoundError(f"File Dataset Test tidak ditemukan: {TEST_CSV_PATH}")
print(f"Loading Dataset Test dari: all_eq_3_0_test.csv")
df_test = pd.read_csv(TEST_CSV_PATH)

# 4. EKSEKUSI INFERENSI
predictions_list = []

print("\nMulai ekstraksi fitur dan prediksi...")
with torch.no_grad():
    for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Predicting Test Set"):
        id_val, subject, gender, ground_truth = row['id'], row['subject'], row['gender'], row['sentence']
        
        # Ekstrak Fitur EEG
        eeg_signal = load_eeg_signal(id_val, subject, gender, config)
        if eeg_signal is None or eeg_signal.shape[0] < config['hop_length']:
            print(f"[WARNING] Skipping ID {id_val}: Sinyal tidak valid atau terlalu pendek.")
            continue
            
        features = compute_hilbert_spectrum(eeg_signal, config)
        
        # Konversi ke Tensor dan tambah dimensi batch (1, Time, Features)
        features_tensor = torch.FloatTensor(features).unsqueeze(0).to(DEVICE)
        
        # Prediksi menggunakan Beam Decoder
        pred_text = beam_decoder.decode(features_tensor)
        
        # Hitung Error Rate (CER)
        cer = compute_cer(ground_truth, pred_text)
        
        predictions_list.append({
            'id': id_val, 
            'subject': subject, 
            'gender': gender,
            'sentence': ground_truth, 
            'prediction': pred_text, 
            'cer': cer
        })

# 5. SIMPAN HASIL PREDIKSI KE CSV
df_result = pd.DataFrame(predictions_list)
df_result.to_csv(OUTPUT_CSV_PATH, index=False)

print("\n" + "=" * 80)
print("✓ PROSES INFERENSI SELESAI")
print(f"Total Baris Diproses  : {len(df_result)}")
print(f"Rata-rata Test CER    : {df_result['cer'].mean():.4f}")
print(f"Hasil Disimpan di     : {OUTPUT_CSV_PATH}")
print("=" * 80)

# Preview 5 data teratas
display(df_result.head())

EEG-to-Text Inference Pipeline (All Subjects | Fixed Hilbert)
[INFO] Menggunakan device: cuda
Loading Tokenizer dari: all_eq_3_0_fixed_hilbert_char_tokenizer_1_1.pkl
Loading Model dari: all_eq_3_0_fixed_hilbert_best_model_1_1.pt
Config berhasil dimuat. Vocab size: 26
Loading Dataset Test dari: all_eq_3_0_test.csv

Mulai ekstraksi fitur dan prediksi...


Predicting Test Set: 100%|██████████| 189/189 [32:38<00:00, 10.36s/it]


✓ PROSES INFERENSI SELESAI
Total Baris Diproses  : 189
Rata-rata Test CER    : 0.7452
Hasil Disimpan di     : d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\all_eq_3_0_fixed_hilbert_test_predictions_1_1.csv


,id,subject,gender,sentence,prediction,cer
0,2_SUB1,SUB1,male,maaf saya terlambat,maaf membuat anda,0.631579
1,3_SUB1,SUB1,male,biar saya bantu ganti baterainya,maaf membuat anda,0.687500
2,15_SUB1,SUB1,male,sebaiknya kita sarapan sekarang,maaf membuat anda,0.806452
3,21_SUB1,SUB1,male,kamu tampak penuh energi hari ini,maaf membuat anda,0.787879
4,22_SUB1,SUB1,male,setelah makan saya merasa sangat kenyang,maaf membuat anda,0.725000
